In [ ]:
%matplotlib widget
import torch
import warnings
import os
import copy
import subprocess
import shlex
import matplotlib.pyplot as plt
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm
from torch.profiler import profile,  ProfilerActivity

os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

from diffSPH.sampling import buildDomainDescription
from diffSPH.modules.adaptiveSmoothingASPH import n_h_to_nH
from diffSPH.plotting import visualizeParticles, updatePlot
from diffSPH.integration import getIntegrator
from diffSPH.util import volumeToSupport
from diffSPH.boundary import sampleDomainSDF
from diffSPH.kernels import Kernel_Scale
from diffSPH.sdf import getSDF, sdfFunctions, operatorDict, sampleSDF
from diffSPH.regions import buildRegion, filterRegion, plotRegions
from diffSPH.modules.timestep import computeTimestep
from diffSPH.schemes.initializers import initializeSimulation, updateBodyParticles
from diffSPH.schemes.deltaSPH import deltaPlusSPHScheme, DeltaPlusSPHSystem
from diffSPH.schema import getSimulationScheme
from diffSPH.enums import *
from exampleUtil import setupExampleSimulation, runSimulation, postProcess
from diffSPH.sampling import sampleDivergenceFreeNoise
import math
from diffSPH.sampling import sampleDivergenceFreeNoise
from diffSPH.modules.particleShifting import solveShifting, shuffleParticles
import numpy as np

In [ ]:
simulationName = 'Flow Past Sphere'
exportName = f'18_flowPastSphere'

L = 2
nx = 64
dx = L / nx
targetDt = 0.001
rho0 = 1
freeSurface = False
band = 0
fps = 50
timeLimit = 10

kernel = KernelType.Wendland4
scheme = SimulationScheme.DeltaSPH
integrationScheme = IntegrationSchemeType.symplecticEuler

device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
dtype = torch.float32
targetNeighbors = n_h_to_nH(4, 2)
c_s = 0.3 * volumeToSupport(dx**2, targetNeighbors, 2) / Kernel_Scale(kernel, 2) / targetDt
exportInterval = 1 / fps
plotInterval = int(math.ceil(exportInterval / targetDt))
timesteps = int(timeLimit / targetDt)

print(c_s)

In [ ]:
dim = 2
domain = buildDomainDescription(l = 2, dim = dim, periodic = True, device = device, dtype = dtype)
# print(domain.min)
domain.min = torch.tensor([-2*L/2,-L/2], device = device, dtype = dtype)
domain.max = torch.tensor([5*L/2,L/2], device = device, dtype = dtype)
# interiorDomain = buildDomainDescription(l = L, dim = dim, periodic = False, device = device, dtype = dtype)
wrappedKernel = kernel
simulator, SimulationSystem, config, integrator = getSimulationScheme(
     scheme, kernel, integrationScheme, 
     1.0, targetNeighbors, domain)
integrationScheme = getIntegrator(integrationScheme)


config['particle'] = {
    'nx': nx + 2 * band,
    'dx': L/nx,
    'targetNeighbors': targetNeighbors,
    'band': band
}
config['fluid'] = {
    'rho0': 1,
    'c_s': c_s
}
config['surfaceDetection']['active'] = freeSurface
config['shifting']['freeSurface'] = freeSurface

In [ ]:
config['diffusion']

In [ ]:
fluid_sdf = lambda x: sampleDomainSDF(x, domain, invert = True)
# domain_sdf = lambda x: sampleDomainSDF(x, interiorDomain, invert = False)
obstacle_sdf = lambda points: sampleSDF(points, lambda x: getSDF('circle')['function'](x, torch.tensor(1/4).to(points.device)), invert = False)

box_sdf = lambda points: sampleSDF(points, lambda x: getSDF('box')['function'](x, torch.tensor([0.5,0.5]).to(points.device)))

inlet_sdf = lambda points: sampleSDF(points, operatorDict['translate'](lambda x: getSDF('box')['function'](x, torch.tensor([L/16,L/2]).to(points.device)), torch.tensor([domain.min[0]+L/16,0]).to(points.device)), invert = False)
inlet_sdf2 = lambda points: sampleSDF(points, operatorDict['translate'](lambda x: getSDF('box')['function'](x, torch.tensor([L/8,L/2]).to(points.device)), torch.tensor([domain.min[0]+L/8,0]).to(points.device)), invert = False)


outlet_sdf = lambda points: sampleSDF(points, operatorDict['translate'](lambda x: getSDF('box')['function'](x, torch.tensor([L/12,L]).to(points.device)), torch.tensor([domain.max[0]-L/12,0]).to(points.device)), invert = False)
outletBuffer_sdf = lambda points: sampleSDF(points, operatorDict['translate'](lambda x: getSDF('box')['function'](x, torch.tensor([L/8,L]).to(points.device)), torch.tensor([domain.max[0]-L/8,0]).to(points.device)), invert = False)


regions = []

# regions.append(buildRegion(sdf = domain_sdf, config = config, type = 'boundary', kind = 'constant'))
regions.append(buildRegion(sdf = obstacle_sdf, config = config, type = 'boundary', kind = 'zero'))
regions.append(buildRegion(sdf = fluid_sdf, config = config, type = 'fluid'))

regions.append(buildRegion(sdf = inlet_sdf, config = config, type = 'inlet', dirichletValues={'velocities': torch.tensor([1,0], device = device, dtype = dtype)}, updateValues = {'densities': 0, 'velocities': torch.tensor([0,0], device = device, dtype = dtype)}))
# regions.append(buildRegion(sdf = inlet_sdf, config = config, type = 'buffer', bufferValues={'densities'}))

# regions.append(buildRegion(sdf = inlet_sdf2, config = config, type = 'forcing', dirichletValues={'velocities': torch.tensor([1,0], device = device, dtype = dtype)}))

regions.append(buildRegion(sdf = outlet_sdf, config = config, type = 'outlet'))
regions.append(buildRegion(sdf = outletBuffer_sdf, config = config, type = 'buffer', bufferValues = ['densities', 'velocities', 'pressures']))

# regions.append(buildRegion(sdf = box_sdf, config = config, type = 'dirichlet', dirichletValues={'densities': 2.0, 'velocities': torch.tensor([1,2], device = device, dtype = dtype), 'pressures': lambda x: torch.where(x[:,0] > 0, 0.0, 1.0)}, updateValues = {'densities': 2.0}))


for region in regions:
    region = filterRegion(region, regions)


In [ ]:


fig, axis = plt.subplots(1, 1, figsize=(8, 5), squeeze=False)
    
plotRegions(regions, axis[0,0], plotFluid = True, plotParticles = False)
axis[0,0].set_aspect('equal')
print(sum([region['particles'].positions.shape[0] for region in regions]))

In [ ]:
from diffSPH.sampling import generateRamp

particleState, config, rigidBodies = initializeSimulation(scheme, config, regions)

particleState.velocities[:,0] = generateRamp(particleState, config) * 1


In [ ]:
exportName = exportName + '2'

In [ ]:
## Setup simulation, everything after this should be the same for all examples
fig, axis, velocityPlot, densityPlot, uidPlot, particleSystem, dt, initialKineticEnergy, initialPotentialEnergy, initialEnergy = setupExampleSimulation(simulationName, scheme, particleState, config, regions, stacked = 'horizontal', figsize = (12, 3), markerSize = 0.15)

imagePrefix = f'./images/{exportName}/'
os.makedirs(imagePrefix, exist_ok = True)
fig.savefig(f'{imagePrefix}frame_{0:05d}.png', dpi = 100)


In [ ]:
fig, axis = plt.subplots(1, 1, figsize=(12, 3), squeeze=False)

# velocityPlot = visualizeParticles(fig, axis[0,0], 
#             particles = particleSystem.systemState, 
#             domain = config['domain'], 
#             quantity = particleSystem.systemState.velocities, 
#             which = 'fluid',
#             mapping = '.x',
#             cmap = 'RdBu_r',
#             visualizeBoth=False,
#             kernel = config['kernel'],
#             plotDomain = False,
#             gridVisualization=False, markerSize=2, streamLines=False, operation='curl', scaling = 'linear', vmin= -7.5, vmax = 7.5)


velocityPlot = visualizeParticles(fig, axis[0,0], 
            particles = particleSystem.systemState, 
            domain = config['domain'], 
            quantity = particleSystem.systemState.velocities, 
            which = 'both',
            mapping = 'L2',
            cmap = 'viridis',
            cbar = False,
            visualizeBoth=False,
            kernel = config['kernel'],
            plotDomain = False,
            gridVisualization=True, markerSize=3, streamLines=False, operation=None, scaling = 'linear', vmin= 0, vmax = 1.5)

axis[0,0].set_xlim(-1.25, 4.5)
axis[0,0].set_ylim(-1., 1.)

axis[0,0].set_xticklabels([])
axis[0,0].set_yticklabels([])
fig.tight_layout()

In [ ]:
fig, axis = plt.subplots(1, 1, figsize=(12, 3), squeeze=False)

# velocityPlot = visualizeParticles(fig, axis[0,0], 
#             particles = particleSystem.systemState, 
#             domain = config['domain'], 
#             quantity = particleSystem.systemState.velocities, 
#             which = 'fluid',
#             mapping = '.x',
#             cmap = 'RdBu_r',
#             visualizeBoth=False,
#             kernel = config['kernel'],
#             plotDomain = False,
#             gridVisualization=False, markerSize=2, streamLines=False, operation='curl', scaling = 'linear', vmin= -7.5, vmax = 7.5)


velocityPlot = visualizeParticles(fig, axis[0,0], 
            particles = particleState, 
            domain = config['domain'], 
            quantity = particleState.UIDs, 
            which = 'both',
            mapping = 'L2',
            cmap = 'turbo',
            cbar = False,
            visualizeBoth=False,
            kernel = config['kernel'],
            plotDomain = False,
            gridVisualization=False, markerSize=5, streamLines=False, operation=None, scaling = 'linear')

axis[0,0].set_xlim(-1.25, 4.5)
axis[0,0].set_ylim(-1., 1.)

axis[0,0].set_xticklabels([])
axis[0,0].set_yticklabels([])
fig.tight_layout()

In [ ]:
particleState.UIDs

In [ ]:
# if outFile is not None:
    # outGroup = outFile.create_group('simulationData')
for i in (tq:=tqdm(range(timesteps))):
    particleSystem, currentState, updates = integrationScheme.function(particleSystem, dt, simulator, config, priorStep = particleSystem.priorStep, verbose = False)

    if i % plotInterval == plotInterval - 1 or i == timesteps - 1:

        rhoMin = particleSystem.systemState.densities.min().detach().cpu().item() / config['fluid']['rho0']
        rhoMean = particleSystem.systemState.densities.mean().detach().cpu().item() / config['fluid']['rho0']
        rhoMax = particleSystem.systemState.densities.max().detach().cpu().item() / config['fluid']['rho0']

        kineticEnergy = 0.5 * particleSystem.systemState.masses / config['fluid']['rho0'] * particleSystem.systemState.densities * torch.linalg.norm(particleSystem.systemState.velocities, dim = -1)**2
        potentialEnergy = None
        if config['gravity']['active']:
            if 'mode' in config['gravity'] and config['gravity']['mode'] == 'potential':
                B = config['gravity']['magnitude']
                potentialEnergy = 0.5 * B**2 * particleSystem.systemState.masses / config['fluid']['rho0'] * particleSystem.systemState.densities * torch.linalg.norm(particleSystem.systemState.positions, dim = -1)**2
                
        combinedEnergy = kineticEnergy + potentialEnergy if potentialEnergy is not None else kineticEnergy
        totalInitialEnergy = (initialEnergy).sum().detach().cpu().item()
        totalEnergy = (combinedEnergy).sum().detach().cpu().item()

        fig.suptitle(f'{simulationName}, ptcls = {particleSystem.systemState.positions.shape[0]}, kernel = {config["kernel"].name}, neighbors = {config["targetNeighbors"]:.2g}, $c_s$ = {config["fluid"]["c_s"]:.1f}\n$\\rho$ = [{rhoMin:.4g} | {rhoMean:.4g} | {rhoMax:.4g}], $E_0$ = {totalInitialEnergy:.4g}, $E$ = {totalEnergy:.4g}, $\\Delta E$ = {totalEnergy - totalInitialEnergy:.4g}, $t$ = {particleSystem.t:.4g}, $\Delta t$ = {dt:.2e}')
        updatePlot(velocityPlot, particleSystem.systemState, particleSystem.systemState.velocities)
        updatePlot(densityPlot, particleSystem.systemState, particleSystem.systemState.densities)
        updatePlot(uidPlot, particleSystem.systemState, particleSystem.systemState.UIDs)
        # if plotCallbackFn is not None:
            # plotCallbackFn(fig, axis, particleSystem, i)
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(f'{imagePrefix}frame_{i:05d}.png', dpi = 100)

In [ ]:

runSimulation(simulationName, particleSystem, integrationScheme, simulator, timesteps, dt, config, fig, axis, velocityPlot, densityPlot, uidPlot, initialEnergy, imagePrefix, plotInterval)

print('Finished simulation')
## Post process simulation
postProcess(imagePrefix, fps, timesteps, exportName)
## Cleanup